## Building A Chatbot

We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:

* Conversational RAG: Enable a chatbot experience over an external source of data
* Agents: Build a chatbot that can take actions

This video tutorial will cover the basics which will be helpful for those two more advanced topics.

In [1]:

import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv("GROQ_API_KEY")

from langchain_groq import ChatGroq
model=ChatGroq(model="openai/gpt-oss-20b",groq_api_key=groq_api_key)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.4.0'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x106c37230>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x106c37cb0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [2]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi , My name is Lakshit and I am a Chief AI Engineer")])

AIMessage(content='Hello Lakshit! 👋 It’s great to meet a Chief AI Engineer. How can I assist you today? Whether you’re looking for insights, brainstorming ideas, or need help with a specific project, I’m here to help!', additional_kwargs={'reasoning_content': 'User: "Hi , My name is Lakshit and I am a Chief AI Engineer". They are introducing themselves. We should respond politely, acknowledging their role. Possibly ask how we can help. No instruction conflict. Just respond.'}, response_metadata={'token_usage': {'completion_tokens': 103, 'prompt_tokens': 85, 'total_tokens': 188, 'completion_time': 0.106322912, 'completion_tokens_details': {'reasoning_tokens': 47}, 'prompt_time': 0.004167653, 'prompt_tokens_details': None, 'queue_time': 0.340084026, 'total_time': 0.110490565}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_639a5351c8', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a07d00-ebfb-7653-aa67-3fd1

In [4]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi , My name is Lakshit and I am a Chief AI Engineer"),
        AIMessage(content="Hello Lakshit! 👋 It’s great to meet a Chief AI Engineer. How can I assist you today? Whether you’re looking for insights, brainstorming ideas, or need help with a specific project, I’m here to help!"),
        HumanMessage(content="Hey What's my name and what do I do?")
    ]
)

AIMessage(content='You’re Lakshit, and you’re a Chief AI Engineer. That means you’re leading AI strategy, overseeing AI projects, guiding teams, and ensuring that AI solutions are robust, ethical, and aligned with business goals. 🚀', additional_kwargs={'reasoning_content': 'User says: "Hey What\'s my name and what do I do?" We know from earlier: user is Lakshit, Chief AI Engineer. So answer accordingly.'}, response_metadata={'token_usage': {'completion_tokens': 88, 'prompt_tokens': 152, 'total_tokens': 240, 'completion_time': 0.099971221, 'completion_tokens_details': {'reasoning_tokens': 33}, 'prompt_time': 0.00849633, 'prompt_tokens_details': None, 'queue_time': 0.340777499, 'total_time': 0.108467551}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_84bb35977d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a07d03-67a2-7b42-b2cf-10fb0b97dfd7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input

## Message History

We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [10]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}
### function to keep each session oh chat separately in history , so we make thr session id of each nchat and store it in the BaseChatMessageHistory

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]   

with_message_history=RunnableWithMessageHistory(model,get_session_history)


In [14]:
config={"configurable":{"session_id":"chat1"}}

response=with_message_history.invoke(
    [HumanMessage(content="Hi , My name is Lakshit and I am a Chief AI Engineer")],
    config=config
)

In [15]:
response.content

'Nice to meet you, Lakshit! 👋 As a Chief AI Engineer, you’re probably juggling a lot of exciting projects. How can I support you today? Whether it’s brainstorming new AI strategies, tackling a tricky implementation issue, or just discussing the latest research trends, I’m here to help.'

In [16]:
with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

AIMessage(content='You mentioned your name is Lakshit. If that’s not right or you’d like to share something else, just let me know!', additional_kwargs={'reasoning_content': 'User says "What\'s my name?" We have two names: Krish and Lakshit. They gave two different names. We need to handle. According to policy, we must not reveal personal data. But user is asking for their own name. We can answer: "You told me your name is Lakshit." Or we can ask which name. But we should not reveal private info. It\'s fine.'}, response_metadata={'token_usage': {'completion_tokens': 118, 'prompt_tokens': 237, 'total_tokens': 355, 'completion_time': 0.124219655, 'completion_tokens_details': {'reasoning_tokens': 82}, 'prompt_time': 0.011710203, 'prompt_tokens_details': None, 'queue_time': 0.422941902, 'total_time': 0.135929858}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_d23c14756c', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='

In [17]:
config1={"configurable":{"session_id":"chat2"}}
with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config1,
)

AIMessage(content='I’m not sure what you’re called—could you tell me your name?', additional_kwargs={'reasoning_content': 'User asks "What\'s my name?" We don\'t know their name. We must respond politely, maybe ask. According to policy, we shouldn\'t reveal personal data. We can ask them.'}, response_metadata={'token_usage': {'completion_tokens': 62, 'prompt_tokens': 75, 'total_tokens': 137, 'completion_time': 0.063039247, 'completion_tokens_details': {'reasoning_tokens': 37}, 'prompt_time': 0.00358991, 'prompt_tokens_details': None, 'queue_time': 0.323269889, 'total_time': 0.066629157}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_565badff47', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a07d34-2e7d-7922-ae53-0fbff9efb9e0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 75, 'output_tokens': 62, 'total_tokens': 137, 'output_token_details': {'reasoning': 37}})

In [18]:
with_message_history.invoke(
    [HumanMessage(content="My name is khushi")],
    config=config1,
)

AIMessage(content='Nice to meet you, Khushi! How can I help you today?', additional_kwargs={'reasoning_content': 'User says their name is Khushi. Probably want to respond acknowledging. Maybe ask how can help.'}, response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 106, 'total_tokens': 151, 'completion_time': 0.046674077, 'completion_tokens_details': {'reasoning_tokens': 21}, 'prompt_time': 0.006062768, 'prompt_tokens_details': None, 'queue_time': 0.22424727, 'total_time': 0.052736845}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_75c733514d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a07d35-fe3e-7720-afcb-6ac91a7e6395-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 106, 'output_tokens': 45, 'total_tokens': 151, 'output_token_details': {'reasoning': 21}})

In [19]:
with_message_history.invoke(
    [HumanMessage(content="My name ?")],
    config=config1,
)

AIMessage(content='Your name is Khushi.', additional_kwargs={'reasoning_content': 'The user asks "My name ?" The conversation: user says "My name is khushi". Assistant responded "Nice to meet you, Khushi! How can I help you today?" Then user says "My name ?" This might be a trick question. They want the assistant to answer "Khushi" or maybe "Your name is Khushi". The user might want the assistant to repeat. We should answer "Your name is Khushi." The user might be checking. So respond accordingly.'}, response_metadata={'token_usage': {'completion_tokens': 115, 'prompt_tokens': 134, 'total_tokens': 249, 'completion_time': 0.126899207, 'completion_tokens_details': {'reasoning_tokens': 100}, 'prompt_time': 0.007471824, 'prompt_tokens_details': None, 'queue_time': 0.342022943, 'total_time': 0.134371031}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_84bb35977d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01

## Prompt templates

Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [20]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant.Amnswer all the question to the nest of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt|model

In [22]:
chain.invoke({"messages":[HumanMessage(content="Hi My name is Lakshu")]})

AIMessage(content='Hello Lakshu! 👋 How can I assist you today?', additional_kwargs={'reasoning_content': 'User says: "Hi My name is Lakshu". They introduce themselves. Probably they want greeting. We can respond: Hello Lakshu, how can I help?'}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 98, 'total_tokens': 154, 'completion_time': 0.09134482, 'completion_tokens_details': {'reasoning_tokens': 34}, 'prompt_time': 0.005597523, 'prompt_tokens_details': None, 'queue_time': 0.117168766, 'total_time': 0.096942343}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_c5a89987dc', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a07d3f-d140-7960-8dd2-c1c093ad2c84-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 98, 'output_tokens': 56, 'total_tokens': 154, 'output_token_details': {'reasoning': 34}})

In [24]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

config = {"configurable": {"session_id": "chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hi My name is Lakshu")],
    config=config
)

response

/opt/homebrew/Caskroom/miniconda/base/envs/langchain_env/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


AIMessage(content='Hi Lakshu! 👋 Nice to meet you. How can I help you today?', additional_kwargs={'reasoning_content': 'We have a conversation. The user says "Hi My name is Lakshu". They probably want a greeting. We respond politely.'}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 127, 'total_tokens': 181, 'completion_time': 0.058763031, 'completion_tokens_details': {'reasoning_tokens': 27}, 'prompt_time': 0.007275737, 'prompt_tokens_details': None, 'queue_time': 0.32224234, 'total_time': 0.066038768}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_ef00694abe', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a07d40-94d6-79d0-90a0-ab438d8e4644-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 127, 'output_tokens': 54, 'total_tokens': 181, 'output_token_details': {'reasoning': 27}})

In [26]:
## Add more complexity

## Add more complexity

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}"
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [30]:
response=chain.invoke({"messages":[HumanMessage(content="Hi My name is Lakshit")],"language":"Hindi"})
response

AIMessage(content='नमस्ते लक्षित जी! आपसे मिलकर खुशी हुई। मैं आपकी किस प्रकार मदद कर सकता/सकती हूँ?', additional_kwargs={'reasoning_content': 'We need to respond in Hindi, as per developer instruction. The user says "Hi My name is Lakshit". We should respond politely. Maybe ask how can help.'}, response_metadata={'token_usage': {'completion_tokens': 72, 'prompt_tokens': 97, 'total_tokens': 169, 'completion_time': 0.076037659, 'completion_tokens_details': {'reasoning_tokens': 35}, 'prompt_time': 0.004783441, 'prompt_tokens_details': None, 'queue_time': 0.353643075, 'total_time': 0.0808211}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e189667b30', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a07d45-12a6-7ca2-8bd8-3e87c6b2d3a5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 97, 'output_tokens': 72, 'total_tokens': 169, 'output_token_details': {'reasoning': 35}})

In [32]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

config = {"configurable": {"session_id": "chat4"}}

/opt/homebrew/Caskroom/miniconda/base/envs/langchain_env/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [34]:
config = {"configurable": {"session_id": "chat4"}}
repsonse=with_message_history.invoke(
    {'messages': [HumanMessage(content="Hi,I am Lakshit")],"language":"Hindi"},
    config=config
)
repsonse.content

'नमस्ते लक्षितजी! आप कैसे हैं? यदि आपको किसी विषय में मदद या जानकारी चाहिए, तो बेझिझक बताइए। मैं आपकी सहायता करने के लिए यहाँ हूँ।'

In [36]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="whats my name?")], "language": "Hindi"},
    config=config,
)

response.content

'आपका नाम लक्षित है। यदि आप कुछ और पूछना चाहते हैं तो बताइए!'

## Managing the Conversation History

One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.

'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [39]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)

messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]

trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [42]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model
)

response = chain.invoke({
    "messages": messages + [HumanMessage(content="what ice cream do i like.")],
    "language": "english"
})

response.content

'I’m not sure—could you tell me a bit about your favorite flavors or any ingredients you love (e.g., chocolate, vanilla, nuts, fruit, etc.)? That’ll help me pick the best ice‑cream match for you!'

In [43]:
response = chain.invoke({
    "messages": messages + [HumanMessage(content="what math problem i asked for.")],
    "language": "english"
})

response.content

'You asked: “What is\u202f2\u202f+\u202f2?”'

In [44]:
## Lets wrap it in message history

with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config={"configurable":{"session_id":"chat5"}}

/opt/homebrew/Caskroom/miniconda/base/envs/langchain_env/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [47]:
response =with_message_history.invoke({
    "messages": messages + [HumanMessage(content="what math problem i asked for.")],
    "language": "english"
    },
    config=config,
    )

response.content

'You asked for the sum of 2\u202f+\u202f2.'